# 第11周 · 周三：特征工程与分析设计——把原始数据变成分析武器

> 难度：⭐⭐⭐⭐ | 定位：掌握特征工程的系统方法论，学会"先写方案再动手"的分析设计纪律
> 前置依赖：W11D2数据清洗与EDA | 后续用途：W11D4-5深度分析与建模
> 面试地位：特征工程是数据分析师和初级数据科学家的核心分水岭——
> 能设计出好特征的人，即使模型选得简单，效果也远超用复杂模型但特征很烂的人

---

## 区块1：核心概念与第一性原理

### 1.1 特征工程的第一性原理

特征工程的本质是：**将原始数据转化为机器能够理解、业务能够解释的信息表示**。

从第一性原理推导为什么特征工程如此重要：

**原命题**：数据分析/建模的目标是从数据中提取规律。
**推论1**：原始数据是"原材料"——它是从业务系统直接产出的，结构是为了存储效率而设计的，不是为了分析。
**推论2**：分析模型是有"理解能力上限"的——线性模型只能理解线性关系，树模型能理解非线性关系但需要明确的特征来分割。
**推论3**：好特征 = 降低模型的"理解难度"。如果一个关系需要100层神经网络才能学到，而你可以用一个特征直接表达这个关系，那就省掉了100层网络的工作量。
**结论**：特征工程是连接"原始数据"和"分析模型"之间的桥梁。桥造得越好，模型越容易到达对岸。

### 1.2 特征工程的三个层次

| 层次 | 定义 | 示例 | 难度 |
|------|------|------|------|
| **L1: 聚合特征** | 从原始记录聚合到分析单元 | sum(amount) per user, count(order_id) per user | ⭐ |
| **L2: 变换特征** | 对已有特征做数学变换 | log(amount), z-score标准化, 分箱(分位数分组) | ⭐⭐ |
| **L3: 衍生特征** | 基于业务逻辑创造新特征 | RFM得分、用户生命周期阶段、品类偏好向量 | ⭐⭐⭐ |

大多数分析项目做到L2就够了。L3是区分"好分析师"和"优秀分析师"的关键。

### 1.3 特征工程 vs 特征选择

两者经常被混淆，但本质不同：

- **特征工程**：创造新的特征（加法）——"我从原始数据中提取了什么新的信息？"
- **特征选择**：筛选重要的特征（减法）——"哪些特征对预测目标真正有用？"

好的流程是：先广泛地做特征工程（创造30-50个候选特征），再用特征选择筛选出10-15个最有用的。

### 1.4 分析方案的"脚手架"——为什么需要先写方案

分析师最大的时间黑洞是"漫无目的地探索数据"。写分析方案的作用不是限制你的探索，而是确保你的探索有方向：

```
没有方案的分析:
  看看这个 → 有点意思 → 再看看那个 → 好像也有关 → 写段代码试试 →
  不对 → 换一个方向 → 2小时过去了 → 产出一个不知道有没有用的图

有方案的分析:
  核心问题是什么 → 用哪个框架 → 需要哪些表 → 产出的形式是什么 →
  开始做 → 1小时后核心部分完成 → 用剩下的时间做探索性的补充分析
```

### 1.5 面试高频考点

**面试官常问**：
- "你做过的最有价值的特征是什么？"
- "如何处理高基数(high-cardinality)的分类变量？"
- "如何评估一个特征的好坏？"
- "特征工程和特征选择的区别是什么？"

**回答要点**：
- 最有价值的特征 = 业务含义清晰 + 预测能力强 + 容易解释
- 高基数分类变量：目标编码(target encoding)、频次编码、分组合并（保留TopN，其他归为"其他"）
- 评估特征好坏：IV值（二分类目标）、互信息、特征重要性（树模型）、稳定性（PSI）
- 特征工程做加法（创建特征）、特征选择做减法（筛选特征）

## 区块2：知识点详细拆解

### 2.1 聚合特征——从订单级到用户级

**为什么重要**：这是最基础也是最常用的特征工程。把"一行一个事件"变成"一行一个分析单元"。

**聚合粒度选择**：
- 用户级(user-level)：一个用户一行，适合用户分群、流失预测
- 品类级(category-level)：一个品类一行，适合商品分析、库存管理
- 时段级(time-level)：一天/一周/一月一行，适合时间序列分析
- 交叉级(cross-level)：一个用户×品类一行，适合个性化推荐

**常用聚合函数**：
```python
user_features = df.groupby('user_id').agg(
    # 基础统计
    order_count=('amount', 'count'),           # 订单数
    total_spend=('amount', 'sum'),             # 总消费
    avg_order=('amount', 'mean'),              # 平均客单价
    max_order=('amount', 'max'),               # 最大单笔消费
    min_order=('amount', 'min'),               # 最小单笔消费
    std_order=('amount', 'std'),               # 消费标准差（波动性）
    # 时间特征
    first_order=('order_date', 'min'),          # 首次下单日期
    last_order=('order_date', 'max'),           # 最后下单日期
    # 品类多样性
    unique_categories=('category', 'nunique'),  # 购买的品类数
    unique_channels=('channel', 'nunique'),     # 使用的渠道数
).reset_index()
```

### 2.2 时间衍生特征——把日期变成有价值的数值

**为什么重要**：原始日期字段本身无法直接建模，必须转化为数值特征。

**时间距离特征**：
```python
ref_date = pd.Timestamp('2026-07-01')  # 分析参考日期
user_features['days_since_first'] = (ref_date - user_features['first_order']).dt.days
user_features['days_since_last'] = (ref_date - user_features['last_order']).dt.days
user_features['active_days_span'] = (user_features['last_order'] - user_features['first_order']).dt.days
```

**时间频率特征**：
```python
# 月均购买频率（避免新用户偏差——注册才3天的人不能和有300天历史的人直接比）
user_features['active_days_span'] = user_features['active_days_span'].clip(lower=1)  # 避免除0
user_features['order_freq_monthly'] = user_features['order_count'] / user_features['active_days_span'] * 30
# 平均购买间隔（天）
user_features['avg_interval'] = user_features['active_days_span'] / (user_features['order_count'] - 1 + 1e-5)
```

**时间模式特征**：
```python
# 是否周末下单多 → 周末型用户
weekend_orders = df[df['order_date'].dt.dayofweek >= 5].groupby('user_id').size()
user_features['weekend_ratio'] = weekend_orders / user_features['order_count']

# 最近活跃趋势 → 最近30天订单占全部订单的比例
recent_cutoff = ref_date - pd.Timedelta(days=30)
recent_orders = df[df['order_date'] >= recent_cutoff].groupby('user_id').size()
user_features['recent_order_ratio'] = recent_orders / user_features['order_count']
```

### 2.3 RFM特征——经典但绝不落伍

**为什么重要**：RFM（Recency-Frequency-Monetary）是数据驱动用户运营的始祖级框架，1970年代起源于直邮营销，至今仍是用户分层的首选方法。

**RFM的精确定义**：

| 维度 | 定义 | 业务含义 | 计算 |
|------|------|---------|------|
| R (Recency) | 距最后一次消费的天数 | 流失风险 | days_since_last |
| F (Frequency) | 消费频次 | 忠诚度 | order_count |
| M (Monetary) | 消费金额 | 用户价值 | total_spend |

**RFM打分的两种方法**：

**方法1：二分法（简单直观）**
```python
# 高于中位数=高分, 低于=低分
r_median = user_features['days_since_last'].median()
f_median = user_features['order_count'].median()
m_median = user_features['total_spend'].median()

user_features['R_score'] = (user_features['days_since_last'] <= r_median).astype(int)  # R越低越好
user_features['F_score'] = (user_features['order_count'] >= f_median).astype(int)
user_features['M_score'] = (user_features['total_spend'] >= m_median).astype(int)
```

**方法2：五分位法（更精细）**
```python
# 按20%分位数分成1-5分
user_features['R_score_5'] = pd.qcut(user_features['days_since_last'], 5, labels=[5,4,3,2,1])  # R越小越好
user_features['F_score_5'] = pd.qcut(user_features['order_count'], 5, labels=[1,2,3,4,5])
user_features['M_score_5'] = pd.qcut(user_features['total_spend'], 5, labels=[1,2,3,4,5])
```

**RFM八类用户**：
```
R F M
1 1 1 = 重要价值客户（全高）
0 1 1 = 重要挽留客户（R低=很久没来，但F和M高）
1 0 1 = 重要发展客户（R高、F低=最近来过但频次低、M高）
1 1 0 = 重要保持客户（R高、F高、M低=来得多但花得少）
0 0 1 = 一般价值客户
0 1 0 = 一般挽留客户
1 0 0 = 新客户（最近来过但F和M都低）
0 0 0 = 流失客户（全低）
```

### 2.4 分类变量的编码策略

**为什么重要**：分类变量不能直接输入模型，必须编码。错误的编码策略可能导致维度爆炸或信息丢失。

| 编码方法 | 适用场景 | 优缺点 |
|---------|---------|--------|
| One-Hot | 低基数(<10类) | 简单但会产生大量稀疏列 |
| Label Encoding | 有序分类(如学历) | 保留了顺序信息 |
| Frequency Encoding | 高基数分类 | 用频次替代类别，信息压缩 |
| Target Encoding | 监督学习 | 用目标均值替代类别，易过拟合 |
| Embedding | 超高基数 | 需要神经网络，信息保留最好 |

**实践中的选择**：
```python
# One-Hot编码
category_dummies = pd.get_dummies(df['category'], prefix='cat')

# Frequency编码
freq_map = df['category'].value_counts(normalize=True)
df['category_freq'] = df['category'].map(freq_map)

# Target编码（以amount为例）
target_map = df.groupby('category')['amount'].mean()
df['category_target_enc'] = df['category'].map(target_map)
```

### 2.5 数值特征的变换与标准化

**为什么重要**：很多模型（线性模型、KNN、神经网络、PCA）对特征尺度敏感。不做标准化的话，数值范围大的特征会"主导"模型。

| 变换方法 | 公式 | 适用场景 |
|---------|------|----------|
| Min-Max缩放 | (x-min)/(max-min) | 需要固定范围[0,1] |
| Z-score标准化 | (x-μ)/σ | 需要均值为0、标准差为1 |
| Log变换 | log(x+1) | 右偏分布（如金额），压缩大值 |
| Box-Cox变换 | (x^λ-1)/λ | 自动寻找最优参数 |
| 分箱(Binning) | 按分位数分组 | 对异常值完全免疫 |

```python
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PowerTransformer

# Z-score标准化
scaler = StandardScaler()
df['amount_zscore'] = scaler.fit_transform(df[['amount']])

# Log变换（处理右偏分布）
df['amount_log'] = np.log1p(df['amount'])  # log(1+x) 避免log(0)

# 分箱（稳健处理异常值）
df['amount_bin'] = pd.qcut(df['amount'], 10, labels=False, duplicates='drop')
```

### 2.6 交互特征——让模型看到"组合拳"

**为什么重要**：两个单独存在都"没用"的特征，组合起来可能是最强特征。

```python
# 乘积交互
user_features['avg_spend_per_day'] = user_features['total_spend'] / user_features['active_days_span']

# 比率交互
user_features['spend_vs_avg'] = user_features['total_spend'] / user_features['total_spend'].mean()

# 多项式交互
user_features['order_count_sq'] = user_features['order_count'] ** 2

# 类别间的交叉比率
# 如：数码品类消费占比 = 数码消费 / 总消费
```

### 2.7 分析方案设计模板

**为什么重要**：标准化的方案设计能帮你快速启动任何分析项目。

```
分析方案模板 V2.0
━━━━━━━━━━━━━━━━━━━━━━
1. 核心问题（1句话）
   例：新用户注册后的7天内，从"浏览商品"到"完成首单"的转化率
   各环节是多少？哪个环节流失最大？

2. 分析框架（选1-2个）
   □ AARRR  □ RFM  □ 同期群  □ 漏斗  □ 归因  □ 其他

3. 关键拆解维度
   □ 品类  □ 渠道  □ 新老用户  □ 时间  □ 地域  □ 设备

4. 核心指标定义
   指标1：________________ 口径：________________
   指标2：________________ 口径：________________

5. 所需聚合表
   表1：user_features（用户级，含RFM+行为特征）
   表2：daily_metrics（日期级，含GMV+订单数+用户数）
   表3：category_metrics（品类级，含GMV+复购率+客单价）

6. 分析方法清单
   - RFM分群 → 用户价值分层
   - 漏斗分析 → 转化流失诊断
   - 同期群分析 → 留存趋势对比
   - 统计检验 → 假设验证

7. 预期产出
   产出1：________________
   产出2：________________
   产出3：________________
```

## 区块3：动手练习

### 步骤0：加载数据并构造完整的用户级特征表

In [ ]:
# 步骤0：构造完整的用户级特征表
import numpy as np
import pandas as pd
np.random.seed(42)

# 重新生成数据集（基于Day1的结构）
n_orders = 50000; n_users = 1000
start_date, end_date = '2026-01-01', '2026-06-30'
date_range = pd.date_range(start_date, end_date)

df_orders = pd.DataFrame({
    'order_id': range(1, n_orders + 1),
    'user_id': np.random.choice(range(1, n_users + 1), n_orders),
    'order_date': pd.to_datetime(np.random.choice(date_range, n_orders)),
    'category': np.random.choice(['服装', '数码', '食品', '美妆', '家居'], n_orders,
                                  p=[0.30, 0.15, 0.25, 0.20, 0.10]),
    'amount': np.round(np.random.lognormal(4.0, 0.8, n_orders), 2),
    'channel': np.random.choice(['organic', 'paid', 'social', 'referral'], n_orders,
                                 p=[0.40, 0.25, 0.20, 0.15]),
})

print(f"原始数据: {len(df_orders)}条订单, {df_orders['user_id'].nunique()}个用户")
print(f"列: {list(df_orders.columns)}")

### 步骤1：L1聚合特征——基础用户画像

In [ ]:
# 步骤1：L1聚合特征——从订单级到用户级
print("=" * 60)
print("L1: 聚合特征工程")
print("=" * 60)

# 1.1 基础聚合
user_feat = df_orders.groupby('user_id').agg(
    order_count=('amount', 'count'),
    total_spend=('amount', 'sum'),
    avg_order=('amount', 'mean'),
    max_order=('amount', 'max'),
    std_order=('amount', 'std'),
    first_order=('order_date', 'min'),
    last_order=('order_date', 'max'),
    favorite_category=('category', lambda x: x.mode()[0] if not x.mode().empty else '未知'),
    unique_categories=('category', 'nunique'),
    unique_channels=('channel', 'nunique'),
).reset_index()

# 填充std_order的NaN（只有1个订单的用户std=NaN）
user_feat['std_order'] = user_feat['std_order'].fillna(0)

print(f"聚合后: {len(user_feat)}个用户, {len(user_feat.columns)}个特征")
print(f"\n特征列表:")
for col in user_feat.columns:
    print(f"  - {col}: {user_feat[col].dtype}")
print(f"\n前5行预览:")
print(user_feat.head())

### 步骤2：L2-L3衍生特征——RFM + 行为特征

In [ ]:
# 步骤2：L2-L3衍生特征——时间特征 + 行为特征 + RFM
print("=" * 60)
print("L2-L3: 衍生特征工程")
print("=" * 60)

ref_date = pd.Timestamp('2026-07-01')

# 2.1 RFM特征
user_feat['R_days'] = (ref_date - user_feat['last_order']).dt.days  # Recency: 距上次购买天数
user_feat['F_count'] = user_feat['order_count']                      # Frequency: 购买次数
user_feat['M_total'] = user_feat['total_spend']                      # Monetary: 总消费

# 2.2 时间衍生特征
user_feat['active_days'] = (user_feat['last_order'] - user_feat['first_order']).dt.days
user_feat['active_days'] = user_feat['active_days'].clip(lower=1)
user_feat['order_freq_monthly'] = user_feat['order_count'] / user_feat['active_days'] * 30
user_feat['avg_interval_days'] = user_feat['active_days'] / (user_feat['order_count'] - 1 + 1e-5)

# 2.3 消费行为特征
user_feat['cv_spend'] = user_feat['std_order'] / (user_feat['avg_order'] + 1e-5)  # 消费波动系数
user_feat['spend_per_day'] = user_feat['total_spend'] / user_feat['active_days']  # 日平均消费

# 2.4 RFM打分（二分法）
r_median, f_median, m_median = (
    user_feat['R_days'].median(),
    user_feat['F_count'].median(),
    user_feat['M_total'].median()
)

user_feat['R_high'] = (user_feat['R_days'] <= r_median).astype(int)
user_feat['F_high'] = (user_feat['F_count'] >= f_median).astype(int)
user_feat['M_high'] = (user_feat['M_total'] >= m_median).astype(int)

# 组合RFM Score (用4/2/1权重编码)
user_feat['rfm_score'] = (user_feat['R_high'] * 4 +
                           user_feat['F_high'] * 2 +
                           user_feat['M_high'] * 1)

# 2.5 用户标签
rfm_labels = {
    7: '重要价值', 6: '重要挽留', 5: '重要发展', 4: '流失高价值',
    3: '一般价值', 2: '一般挽留', 1: '新客户', 0: '流失客户'
}
user_feat['user_segment'] = user_feat['rfm_score'].map(rfm_labels)

print("特征工程完成！")
print(f"\n[新增衍生特征]")
print(f"  R_days: 距上次购买天数 (min={user_feat['R_days'].min()}, median={user_feat['R_days'].median():.0f}, max={user_feat['R_days'].max()})")
print(f"  F_count: 购买次数 (min={user_feat['F_count'].min()}, median={user_feat['F_count'].median():.0f}, max={user_feat['F_count'].max()})")
print(f"  M_total: 总消费 (min={user_feat['M_total'].min():.0f}, median={user_feat['M_total'].median():.0f}, max={user_feat['M_total'].max():.0f})")
print(f"  order_freq_monthly: 月均购买频次")
print(f"  spend_per_day: 日消费力")
print(f"  rfm_score: RFM综合得分(0-7)")
print(f"  user_segment: 用户分层标签")

print(f"\n[RFM分层分布]")
segment_dist = user_feat['user_segment'].value_counts()
for seg, cnt in segment_dist.items():
    print(f"  {seg}: {cnt}人 ({cnt/len(user_feat)*100:.1f}%)")

### 步骤3：品类级特征 + 挑战练习

In [ ]:
# 步骤3：品类级特征 + 分析方案设计
print("=" * 60)
print("L3: 品类级特征 & 分析方案设计")
print("=" * 60)

# 3.1 品类级特征
category_feat = df_orders.groupby('category').agg(
    total_gmv=('amount', 'sum'),
    order_count=('order_id', 'count'),
    avg_amount=('amount', 'mean'),
    unique_buyers=('user_id', 'nunique'),
).reset_index()

category_feat['gmv_share'] = category_feat['total_gmv'] / category_feat['total_gmv'].sum() * 100
category_feat['avg_orders_per_user'] = category_feat['order_count'] / category_feat['unique_buyers']
print("\n[品类级特征表]")
print(category_feat.round(2))

# 3.2 填写分析方案
print("\n" + "=" * 60)
print("填写你的分析方案")
print("=" * 60)
print()
print("1. 核心问题:")
print("   平台1000名用户中，哪些是核心资产？用户在哪里流失？")
print("   如何用数据驱动的方式提升用户价值？")
print()
print("2. 分析框架: RFM + AARRR漏斗 + 同期群分析")
print()
print("3. 关键拆解维度: 品类 × 渠道 × 新老用户")
print()
print("4. 核心指标:")
print("   指标1: 7日留存率 = Day7活跃用户数 / Day0注册用户数")
print("   指标2: 用户LTV(生命周期价值) = avg_order × order_freq_monthly × 预计留存月数")
print("   指标3: RFM综合得分分布(识别高价值用户集中度)")
print()
print("5. 分析方法清单:")
print("   ☑ RFM分群 → 用户价值分层")
print("   ☑ 漏斗分析 → 转化流失诊断")
print("   ☑ 同期群分析 → 留存趋势对比")
print("   ☑ 统计检验 → 假设验证(渠道差异、品类差异)")
print()
print("6. 预期产出:")
print("   - RFM用户分层可视化(含各层用户画像)")
print("   - 转化漏斗图(含流失环节标注)")
print("   - 同期群留存矩阵(热力图)")
print("   - 3条可落地的策略建议(含预期效果和度量指标)")
print()
print("分析方案已就绪 → 明天进入深度分析！")

## 区块4：业务场景应用

### 场景1：特征工程救了模型一命

**问题**：用用户的"总消费金额"预测用户是否会流失，AUC只有0.55（几乎随机）。

**诊断**："总消费金额"对于老用户和新用户是完全不同的含义——老用户消费高是因为时间长，不代表不会流失。

**特征工程解决**：
```python
# 替换M_total为"相对消费力"
user_feat['spend_per_active_day'] = user_feat['total_spend'] / user_feat['active_days']
user_feat['recent_spend_ratio'] = recent_30d_spend / user_feat['total_spend']
```

**效果**：
- total_spend的AUC: 0.55
- spend_per_active_day的AUC: 0.68
- recent_spend_ratio的AUC: 0.74（最近30天消费占比——花钱越来越少的用户确实在流失）

**教训**：原始特征往往带有"时间偏差"——老用户的绝对值天然大于新用户。要用比率/频率特征来消除这个偏差。

### 场景2："最喜欢的品类"这个特征值多少钱？

**问题**：定义一个"最喜欢的品类"——是买得最多的品类还是花钱最多的品类？

**分析**：
- 按购买次数最多算：可能选出"食品"（高频低客单价）
- 按花钱最多算：可能选出"数码"（低频高客单价）
- 两者反映的性格完全不同——前者是"日常型用户"，后者是"偶发型大额用户"

**决策**：两个都算。创建3个特征：
```python
# 频次偏好
user_feat['fav_by_freq'] = df.groupby('user_id')['category'].agg(lambda x: x.mode()[0])
# 金额偏好
user_feat['fav_by_spend'] = df.groupby('user_id').apply(
    lambda g: g.groupby('category')['amount'].sum().idxmax()
)
# 品类集中度（多样化程度）
user_feat['category_hhi'] = df.groupby(['user_id', 'category']).size().groupby('user_id').apply(
    lambda x: (x / x.sum()).pow(2).sum()  # HHI指数，越高越集中
)
```

### 场景3：分析方案救了你

**问题**：周三下午4点，已经做了3小时分析，但感觉在做无用功——做了很多图但不知道要说明什么。

**诊断**：没有写方案就开始动手了。

**纠正步骤**：
1. 关掉所有代码和图表
2. 拿出一张白纸，写下："这个分析最终要回答的1个核心问题是什么？"
3. 写下："回答这个问题需要3个关键图表/数字是什么？"
4. 只做这3个图表，其他的都是附录

**教训**：先写方案的成本是30分钟。不写方案的成本是一整天的低效工作。这个交易非常划算。

## 区块5：分级检验清单

### 初级检验（必须能独立完成的）
- [ ] 能从原始订单表聚合出用户级特征表
- [ ] 能计算RFM的三个维度并做二分打分
- [ ] 能用pd.qcut做分位数分箱
- [ ] 能填写完整的分析方案模板

### 中级检验（能解释给同事听的原理）
- [ ] 能解释为什么RFM中的R用中位数切断，而不是固定天数（如30天）
- [ ] 能解释特征工程的三个层次（L1聚合/L2变换/L3衍生）的区别
- [ ] 能选择正确的分类变量编码策略（One-Hot vs Frequency vs Target）
- [ ] 能从EDA发现中推导出应该创建哪些特征

### 高级检验（能举一反三的迁移能力）
- [ ] 能设计出"时间归一化"的特征（消除用户注册时长的影响）
- [ ] 能评估特征稳定性（PSI指标）和特征重要性
- [ ] 能在"过拟合风险"和"特征丰富度"之间做平衡
- [ ] 能写出生产级的特征工程pipeline（可复用、可维护）

**答案提示**：
- R用中位数切断：因为R的分布通常是右偏的，中位数比均值更稳健
- 时间归一化：用比率特征代替绝对值特征（如月均消费代替总消费）
- PSI = Σ(actual% - expected%) × ln(actual% / expected%)——衡量特征分布是否随时间漂移

## 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 前置依赖** | W11D2 EDA | EDA中的发现直接转化为今天的特征 |
| **← 前置依赖** | W3D7 Pandas聚合操作 | groupby+agg是特征工程的基础操作 |
| **→ 后续用途** | W11D4-5建模 | 今天的特征表是明天建模的输入 |
| **→ 后续用途** | W8(机器学习周) | 同样的特征工程方法用于预测模型 |
| **业务连接** | 用户运营 | RFM分层直接指导用户运营策略 |

### 横向连接：特征工程的跨领域应用

**与机器学习的连接**：
- 同样的特征工程方法直接用于sklearn的模型训练
- 特征选择（如SelectKBest、RFECV）筛选最优特征子集

**与AB实验的连接**：
- 特征可以用于定义实验的分层抽样条件
- 特征可以用于识别实验中最敏感的亚群

**与数据仓库的连接**：
- 好的特征应该沉淀到数据仓库中，作为"特征集市"供后续分析复用
- 避免每次分析都重新计算同样的特征

## 区块7：当日小结

> **特征工程的本质是把原始数据转化为分析模型能理解的信息表示。三个层次：L1聚合（groupby+agg）、L2变换（标准化/分箱/编码）、L3衍生（RFM/时间模式/行为漏斗）。RFM是经典但绝不落伍的用户分层框架——Recency衡量流失风险、Frequency衡量忠诚度、Monetary衡量用户价值。分析方案是效率的保证：先花30分钟写方案，能省掉后面一整天的方向错误。**

**三条金句带回家**：
1. 好特征 > 好模型。特征工程做得好，线性回归也能打败深度学习；特征工程做得烂，深度学习也救不回来
2. RFM三问：最近什么时候来过？（R）多久来一次？（F）每次花多少钱？（M）——三个问题回答了用户价值的80%
3. "绝对值特征"是陷阱——老用户的绝对值天然大于新用户。用比率/频率特征消除时间偏差，用per-day替代total

### 课后拓展

1. **练习**：在RFM三分法之外，尝试用RFM五分位法重新打分，对比两种方法的用户分层差异

2. **思考**：除了RFM，还有哪些维度值得做成特征？（提示：社交维度——是否推荐过新用户？内容维度——偏好哪种类型的内容？）

3. **预习**：明天将用今天的特征表做深度分析——RFM分层洞察、转化漏斗诊断、同期群留存对比

4. **面试准备**：准备回答"你做过的最有价值的特征是什么"——选一个你今天创建的特征，能说清楚：业务含义 → 为什么有用 → 对分析结论的影响